# Train Implicit Q-Learning (IQL)

## 0. Setup

Mount google drive

In [ ]:
from google.colab import drive
drive.mount('/gdrive')

Link and download libraries

In [ ]:
root = "/gdrive/MyDrive/Colab Notebooks/EE405_IQL"
!ln -s "{root}/iql" ./
!ln -s "{root}/teleoperation_dataset"

!pip install tensorboard

# Using RGB images
# !git clone https://github.com/facebookresearch/r3m
# !cd r3m; pip install -e .

## 1. Import libraries

In [ ]:
import os
import sys

from tqdm import trange

sys.path.append("./iql")
from iql.log import Logger
from iql.IQL import IQL
from iql.utils import ReplayBuffer, make_dir

# sys.path.append("./r3m")
# import r3m

## 2. Train IQL

In [ ]:
# Base directories
WORK_DIR = "iql/results/"
DATA_DIR = "teleoperation_dataset/jetrover-pickup-cube/"

make_dir(WORK_DIR)
make_dir(os.path.join(WORK_DIR, "models"))

# Hyperparameters
expectile = 0.7
temperature = 3.0
tau = 0.005
discount = 0.99
max_timesteps = 5e5
save_freq = 1e5
batch_size = 256

# Dataset
replay_buffer = ReplayBuffer(DATA_DIR)
state_dim = replay_buffer.state_dim
action_dim = replay_buffer.action_dim

policy = IQL(state_dim=state_dim,
                action_dim=action_dim,
                expectile=expectile,
                discount=discount,
                tau=tau,
                temperature=temperature)

logger = Logger(WORK_DIR)

# Train
for t in trange(int(max_timesteps)):
    policy.train(replay_buffer, batch_size, logger=logger)
    if (t + 1) % save_freq == 0:
        policy.save(os.path.join(WORK_DIR, "models"))

In [ ]:
# Tensorboard
%load_ext tensorboard
%tensorboard --logdir {WORK_DIR}/tb/